# Vacua Storage

**What's in this notebook?** This notebook demonstrates how to generate, store, query, and curate flux vacuum solutions using the `CYDatabase` vacua storage system. All examples use **real SUSY vacua** computed with jaxvacua's solvers.

**What you will learn:**

1. Generate SUSY vacua with four ISD sampling modes, random sampling, and flux bounding
2. Store vacua with method labels, tags, and run IDs
3. Query and filter stored vacua by method, model, and custom criteria
4. Designate validated solutions as permanent records
5. Work with multiple models (local and from HuggingFace)

**Prerequisites:** Notebooks 2 (overview), 5 (finding flux vacua), 7 (ISD sampling).

## 1. Setup

In [ ]:
import warnings
import tempfile
import numpy as np

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

from scipy.optimize import root
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    "figure.dpi": 200, "font.size": 10, "axes.labelsize": 11,
    "figure.figsize": (7, 3), "legend.frameon": False,
    "font.family": "serif", "xtick.direction": "in", "ytick.direction": "in",
})

import jaxvacua as jvc
from jaxvacua.database import CYDatabase

warnings.filterwarnings('ignore')

# Create a temporary database for this demo (cleaned up at the end)
tmpdir = tempfile.mkdtemp(prefix="jaxvacua_demo_")
db = CYDatabase.from_local(tmpdir)
print(f"Demo database: {db.cache_dir}")

## 2. Model A: CP$[1,1,1,6,9]$ at $h^{2,1}=2$

We use the degree-18 hypersurface in $\mathbb{CP}[1,1,1,6,9]$, loaded via the local `model_ID=1` path.

In [ ]:
model_A = jvc.FluxVacuaFinder(h12=2, model_ID=1, model_type="KS", maximum_degree=2)
model_A.lcs_tree.a_matrix = jnp.array([[4.5, 1.5], [1.5, 0.]])
sampler = jvc.data_sampler(model_A,flux_bounds=[-5,5])

print(f"h12 = {model_A.h12}, n_fluxes = {model_A.n_fluxes}")
print(f"D3 tadpole = {model_A.D3_tadpole}")

### Helper: solve and collect

We define a small helper that takes ISD initial guesses, solves `$D_i W = 0$` via `scipy.optimize.root`, and returns the converged solutions.

In [ ]:
def solve_from_guesses(model, z0, tau0, fluxes0, tol=1e-10, residual_tol=1e-6):
    """Solve DW=0 for each (z, tau, flux) triple. Return converged solutions."""
    moduli_out, tau_out, flux_out = [], [], []
    for i in range(len(z0)):
        x0 = model._convert_complex_to_real(
            z0[i], jnp.conj(z0[i]), tau0[i], jnp.conj(tau0[i])
        )
        r = root(model.DW_x, x0, args=(fluxes0[i],),
                 jac=model.dDW_x, method="hybr", tol=tol)
        residual = float(np.max(np.abs(r.fun)))
        if residual < residual_tol:
            m, _, t, _ = model._convert_real_to_complex(r.x)
            if not np.any(np.isnan(np.append(m, t))):
                moduli_out.append(m)
                tau_out.append(t)
                flux_out.append(fluxes0[i])
    if len(moduli_out) == 0:
        return jnp.zeros((0, model.h12)), jnp.zeros(0), jnp.zeros((0, 2*model.n_fluxes))
    return jnp.array(moduli_out), jnp.array(tau_out), jnp.array(flux_out)

## 3. Generating vacua with different sampling methods

We generate SUSY vacua using four ISD modes and store each batch separately, labeled by method.

### ISD sampling (4 modes)

ISD (Imaginary Self-Dual) sampling exploits the ISD condition $\star G_3 = \mathrm{i} G_3$ to construct flux vectors that approximately satisfy the F-term equations. Given a random moduli point $(z^a, \tau)$ and half of the flux quanta, the ISD condition fixes the remaining fluxes analytically. After rounding to integers, a short Newton solve finds the nearby exact SUSY vacuum.

Four modes are available:
- **ISD+** / **ISD-**: solve the ISD condition for the upper/lower flux components
- **F**: fix the NSNS-flux $h$ and derive the RR-flux $f$
- **H**: fix the RR-flux $f$ and derive the NSNS-flux $h$

Each mode explores a different region of the flux landscape and yields a different set of vacua.

In [ ]:
Nmax = 276
N_candidates = 50

for isd_mode in ["ISD+", "ISD-", "F", "H"]:
    # Generate ISD initial guesses
    z0, tau0, fluxes0 = sampler.initial_guesses_ISD(
        N=N_candidates, Nmax=Nmax, mode=isd_mode, 
        moduli_sampling_mode="cone", ISD_oversample_factor=2,
        print_progress=True, filter_moduli=True
    )
    
    # Solve for SUSY vacua
    moduli, tau, fluxes = solve_from_guesses(model_A, z0, tau0, fluxes0)
    
    print(f"{isd_mode:5s}: {len(moduli)}/{N_candidates} converged")
    
    # Store in database
    if len(moduli) > 0:
        with db.vacua_writer(model=model_A, method=isd_mode) as w:
            w.append_batch(moduli, tau, fluxes, is_susy=np.ones(len(moduli), dtype=bool))
        print(f"       Stored {w.count} vacua (run_id: {w._run_id[:8]}...)")

### Random uniform sampling

Sample moduli and fluxes uniformly at random, then solve.

In [ ]:
rng = np.random.default_rng(42)
N_random = 100

# Random moduli in the Kahler cone
z_rand = jnp.array(rng.uniform(-0.5, 0.5, (N_random, 2)) + 1j * rng.uniform(2, 5, (N_random, 2)))
tau_rand = jnp.array(rng.uniform(-0.5, 0.5, N_random) + 1j * rng.uniform(1, 10, N_random))
flux_rand = jnp.array(rng.integers(-5, 6, (N_random, 2 * model_A.n_fluxes)), dtype=float)

# Filter by tadpole
sigma = np.array(model_A.periods.sigma)
nf = model_A.n_fluxes
tad = np.array([abs(flux_rand[i, :nf] @ sigma @ flux_rand[i, nf:]) for i in range(N_random)])
mask = (tad > 0) & (tad <= Nmax)
z_rand, tau_rand, flux_rand = z_rand[mask], tau_rand[mask], flux_rand[mask]

moduli_r, tau_r, flux_r = solve_from_guesses(model_A, z_rand, tau_rand, flux_rand)
print(f"random: {len(moduli_r)}/{len(z_rand)} converged (from {N_random} initial)")

if len(moduli_r) > 0:
    with db.vacua_writer(model=model_A, method="random") as w:
        w.append_batch(moduli_r, tau_r, flux_r, is_susy=np.ones(len(moduli_r), dtype=bool))
    print(f"       Stored {w.count} vacua")

### Flux bounding (systematic enumeration)

Use `bounded_fluxes.enumerate_fluxes` with a small $N_\text{max}$ for a quick systematic scan.

In [ ]:
from jaxvacua.flux_bounding import bounded_fluxes

bf = bounded_fluxes(model_A, sampler, Nmax=Nmax)
enum_results = bf.sample_bounded_fluxes(n_sample=50, n_batch=100, n_target=50, n_mod=5, refine=True, verbose=True)

if len(enum_results) > 0:
    # Extract arrays
    m_enum = jnp.array([r["moduli"] for r in enum_results])
    t_enum = jnp.array([r["tau"] for r in enum_results])
    f_enum = jnp.array([r["flux"] for r in enum_results])
    
    with db.vacua_writer(model=model_A, method="flux_bounding") as w:
        w.append_batch(m_enum, t_enum, f_enum, is_susy=np.ones(len(m_enum), dtype=bool))
    print(f"flux_bounding: Stored {w.count} vacua from {len(enum_results)} results")
else:
    print("flux_bounding: No results (try increasing Nmax or n_sample)")

### Perturbatively flat vacua (PFVs)

PFVs are a special class of flux vacua where the polynomial superpotential vanishes exactly along a flat direction $z^a = p^a \tau$. The vacuum is lifted only by exponentially suppressed instanton corrections, leading to very small $|W_0|$.

The PFV is specified by integer vectors $(M^a, K_a)$ satisfying (see [arXiv:2512.17095](https://arxiv.org/abs/2512.17095), Section 6.2):
- $\det(N) \neq 0$ where $N_{ab} = \kappa_{abc} M^c$
- $p^a = (N^{-1} K)^a \in \mathcal{K}_{\widetilde{X}}$ (Kahler cone)
- $K_a p^a = 0$ (orthogonality)

`pfv_to_flux(M, K)` constructs the full flux vector, and `pfv_to_moduli(M, K, tau)` gives the flat-direction starting point.

In [ ]:
# PFV example from arXiv:2501.03984 (Table 1)
M_pfv = jnp.array([-16, 50])
K_pfv = jnp.array([3, -4])
gs = 0.2  # string coupling

flux_pfv = model_A.pfv_to_flux(M_pfv, K_pfv)
tau0_pfv = 1j / gs
z0_pfv = model_A.pfv_to_moduli(M_pfv, K_pfv, tau0_pfv)

print(f"PFV flux: {np.array(flux_pfv, dtype=int)}")
print(f"Starting moduli: z = {z0_pfv}")
print(f"Starting tau:    {tau0_pfv}")

# Newton warm-start (PFV starting point is close but not exact)
z_n, tau_n, _ = model_A.newton_method_flux_vacua(
    z0_pfv, tau0_pfv, flux_pfv,
    step_size_Newton=0.5, tol=1e-12, max_iters=100,
    mode="SUSY", solver_mode="real",
)

# scipy polish
x_n = np.array(model_A._convert_complex_to_real(z_n, jnp.conj(z_n), tau_n, jnp.conj(tau_n)))
r_pfv = root(model_A.DW_x, x_n, args=(flux_pfv,), jac=model_A.dDW_x, method="hybr", tol=1e-12)
residual = float(np.max(np.abs(r_pfv.fun)))

if residual < 1e-6:
    z_pfv, _, tau_pfv, _ = model_A._convert_real_to_complex(jnp.array(r_pfv.x))
    W0 = model_A.superpotential_gauge_invariant(z_pfv, tau_pfv, flux_pfv)
    print(f"\nConverged! |W0_gi| = {abs(complex(W0)):.6e}  (|DW| = {residual:.2e})")
    print(f"z  = {z_pfv}")
    print(f"tau = {tau_pfv}")
    
    # Store as PFV with tags
    with db.vacua_writer(model=model_A, method="PFV",
                         metadata={"M": M_pfv.tolist(), "K": K_pfv.tolist(), "gs": gs}) as w:
        w.append({"moduli": np.array(z_pfv), "tau": complex(tau_pfv),
                  "flux": np.array(flux_pfv)},
                 tags=["racetrack", "small_W0"])
    print(f"Stored PFV vacuum (run_id: {w._run_id[:8]}...)")
else:
    print(f"PFV did not converge (|DW| = {residual:.2e})")

In [ ]:
# Scan over several PFV (M, K) pairs
pfv_pairs = [
    ([-16, 50], [3, -4], 0.2),
    ([-8, 25],  [3, -4], 0.3),
    ([4, -8],   [8, -3], 0.15),
    ([-10, 30], [3, -4], 0.12),
]

pfv_moduli, pfv_tau, pfv_flux = [], [], []
for M_list, K_list, gs_val in pfv_pairs:
    M_i = jnp.array(M_list)
    K_i = jnp.array(K_list)
    fl_i = model_A.pfv_to_flux(M_i, K_i)
    tau_i = 1j / gs_val
    z_i = model_A.pfv_to_moduli(M_i, K_i, tau_i)
    
    try:
        # Newton warm-start + scipy polish
        z_n, tau_n, _ = model_A.newton_method_flux_vacua(
            z_i, tau_i, fl_i, step_size_Newton=0.5, tol=1e-12,
            max_iters=100, mode="SUSY", solver_mode="real",
        )
        x_n = np.array(model_A._convert_complex_to_real(z_n, jnp.conj(z_n), tau_n, jnp.conj(tau_n)))
        r_i = root(model_A.DW_x, x_n, args=(fl_i,), jac=model_A.dDW_x, method="hybr", tol=1e-12)
        residual = float(np.max(np.abs(r_i.fun)))
        
        if residual < 1e-6:
            m_i, _, t_i, _ = model_A._convert_real_to_complex(jnp.array(r_i.x))
            W_i = abs(complex(model_A.superpotential_gauge_invariant(m_i, t_i, fl_i)))
            pfv_moduli.append(m_i)
            pfv_tau.append(t_i)
            pfv_flux.append(fl_i)
            print(f"M={M_list}, K={K_list}, gs={gs_val}: |W0_gi|={W_i:.4e}")
        else:
            print(f"M={M_list}, K={K_list}, gs={gs_val}: did not converge (|DW|={residual:.2e})")
    except Exception as e:
        print(f"M={M_list}, K={K_list}, gs={gs_val}: error — {e}")

if len(pfv_moduli) > 0:
    with db.vacua_writer(model=model_A, method="PFV_scan") as w:
        w.append_batch(jnp.array(pfv_moduli), jnp.array(pfv_tau), jnp.array(pfv_flux),
                       is_susy=np.ones(len(pfv_moduli), dtype=bool),
                       tags=["racetrack"])
    print(f"\nStored {w.count} PFV vacua")

### Summary of SUSY vacua

Let us visualise the vacua collected so far across all sampling methods.

In [ ]:
# Collect all SUSY vacua and plot
all_runs = db.query_vacua(h12=2)
if len(all_runs) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    colors = {"ISD+": "steelblue", "ISD-": "coral", "F": "seagreen", "H": "darkorange",
              "random": "purple", "flux_bounding": "brown", "PFV": "crimson", "PFV_scan": "pink"}
    
    for _, run_row in all_runs.iterrows():
        method = run_row["method"]
        try:
            df = db.load_vacua(run_id=run_row["run_id"])
        except Exception:
            continue
        if len(df) == 0:
            continue
        
        z_arr = np.array(df["moduli"].tolist())
        tau_arr = np.array(df["tau"].tolist())
        c = colors.get(method, "gray")
        
        axes[0].scatter(z_arr[:, 0].real, z_arr[:, 0].imag, s=8, c=c, alpha=0.6, label=method)
        axes[1].scatter(tau_arr.real, tau_arr.imag, s=8, c=c, alpha=0.6)
        
        # Compute |W0| for each vacuum
        for j in range(len(df)):
            row = df.iloc[j]
            try:
                W0 = abs(complex(model_A.superpotential_gauge_invariant(
                    jnp.array(row["moduli"]), complex(row["tau"]), jnp.array(row["flux"]))))
                axes[2].scatter(j, W0, s=8, c=c, alpha=0.6)
            except Exception:
                pass
    
    axes[0].set_xlabel(r"Re$(z^1)$")
    axes[0].set_ylabel(r"Im$(z^1)$")
    axes[0].set_title("Moduli $z^1$")
    # Remove duplicate labels
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    axes[0].legend(by_label.values(), by_label.keys(), fontsize=6, ncol=2)
    
    axes[1].set_xlabel(r"Re$(\tau)$")
    axes[1].set_ylabel(r"Im$(\tau)$")
    axes[1].set_title(r"Axio-dilaton $\tau$")
    
    axes[2].set_xlabel("Vacuum index")
    axes[2].set_ylabel(r"$|\tilde{W}_0|$")
    axes[2].set_yscale("log")
    axes[2].set_title(r"Gauge-invariant $|W_0|$")
    
    fig.tight_layout()
    plt.show()
else:
    print("No vacua stored yet")

## 4. Non-SUSY critical points

We take the SUSY solutions found above and attempt to find nearby non-SUSY critical points by solving $\partial V / \partial \phi = 0$ with perturbed starting points.

In [ ]:
# Load some SUSY solutions to use as seeds
vacua_df = db.query_vacua(h12=2, method="ISD+")
if len(vacua_df) > 0:
    run_id_isd = vacua_df.iloc[0]["run_id"]
    susy_data = db.load_vacua(run_id=run_id_isd)
    
    # Perturb and solve dV=0
    non_susy_moduli, non_susy_tau, non_susy_flux = [], [], []
    rng_ns = np.random.default_rng(99)
    
    for i in range(min(20, len(susy_data))):
        row = susy_data.iloc[i]
        z_s = jnp.array(row["moduli"])
        tau_s = complex(row["tau"])
        fl_s = jnp.array(row["flux"])
        
        # Perturb moduli slightly
        z_pert = z_s + 0.1 * (rng_ns.standard_normal(2) + 1j * rng_ns.standard_normal(2))
        x0 = np.array(model_A._convert_complex_to_real(
            z_pert, jnp.conj(z_pert), tau_s, np.conj(tau_s)
        ))
        
        r = root(model_A.dV_x, x0, args=(fl_s,), jac=model_A.ddV_x, method="hybr", tol=1e-10)
        if r.success and float(np.max(np.abs(r.fun))) < 1e-6:
            m, _, t, _ = model_A._convert_real_to_complex(jnp.array(r.x))
            non_susy_moduli.append(m)
            non_susy_tau.append(t)
            non_susy_flux.append(fl_s)
    
    if len(non_susy_moduli) > 0:
        with db.vacua_writer(model=model_A, method="dV_perturbed") as w:
            w.append_batch(
                jnp.array(non_susy_moduli), jnp.array(non_susy_tau),
                jnp.array(non_susy_flux), is_susy=np.zeros(len(non_susy_moduli), dtype=bool)
            )
        print(f"Non-SUSY: Stored {w.count} critical points")
    else:
        print("Non-SUSY: No converged critical points found")
else:
    print("No SUSY vacua available as seeds")

### Non-SUSY via CriticalPointFinder

The `CriticalPointFinder` class provides a systematic pipeline for finding critical points of the scalar potential $V$. It generates ISD flux candidates, solves $\partial V / \partial \phi = 0$ using Newton or hybrid solvers, and classifies solutions as minima, saddle points, or maxima via the Hessian eigenvalues.

In [ ]:
from jaxvacua.critical_points import CriticalPointFinder

cpf = CriticalPointFinder(model_A, sampler, Nmax=10, noscale=True)

print("Searching for non-SUSY critical points...")
cp_results = cpf.sample_critical_points(
    n_target=10, n_batch=200, max_batches=3,
    isd_mode="ISD-", solver="newton",
    classify=True, verbose=True,
)

if len(cp_results) > 0:
    cp_moduli = jnp.array([r["moduli"] for r in cp_results])
    cp_tau = jnp.array([complex(r["tau"]) for r in cp_results])
    cp_flux = jnp.array([r["flux"] for r in cp_results])
    cp_is_susy = np.array([r.get("is_susy", False) for r in cp_results])
    cp_is_min = np.array([r.get("is_minimum", False) for r in cp_results])

    with db.vacua_writer(model=model_A, method="critical_points") as w:
        w.append_batch(cp_moduli, cp_tau, cp_flux,
                       is_susy=cp_is_susy)
    print(f"Stored {w.count} critical points ")
    print(f"  SUSY: {cp_is_susy.sum()}, non-SUSY: {(~cp_is_susy).sum()}")
    print(f"  Minima: {cp_is_min.sum()}, saddles: {(~cp_is_min).sum()}")
else:
    print("No critical points found (try increasing n_batch or max_batches)")

## 5. Querying and filtering

The vacua storage system supports two levels of filtering:

1. **Catalog-level** (`query_vacua`): fast, in-memory filtering by model identity, method, run_id, SUSY flag, and tadpole bound. Returns metadata without loading solution data.
2. **Data-level**: load solution arrays with `load_vacua(run_id)`, then filter the pandas DataFrame with arbitrary conditions (moduli range, $|W_0|$ threshold, etc.).

### By method

In [ ]:
# Overview of all stored vacua
all_vacua = db.query_vacua(h12=2)
print(f"Total stored vacua for h12=2: {len(all_vacua)}")
print(f"\nMethods: {all_vacua['method'].unique().tolist()}")
print(f"\nCount per method:")
print(all_vacua.groupby("method")["count"].sum().to_string())

### By SUSY flag

In [ ]:
# SUSY only
susy = db.query_vacua(h12=2, is_susy=True)
print(f"SUSY vacua: {len(susy)} runs")

# Non-SUSY only
non_susy = db.query_vacua(h12=2, is_susy=False)
print(f"Non-SUSY vacua: {len(non_susy)} runs")

### Custom filtering on loaded data

`query_vacua` provides fixed filters. For custom criteria (e.g. moduli in a specific region, $|W_0|$ below a threshold), load the data first and filter with pandas.

In [ ]:
# Load all ISD+ vacua
isd_runs = db.query_vacua(method="ISD+")
if len(isd_runs) > 0:
    df = db.load_vacua(run_id=isd_runs.iloc[0]["run_id"])
    print(f"Loaded {len(df)} ISD+ vacua")
    print(f"Columns: {df.columns.tolist()}")
    
    # Custom filter: Im(tau) > 3 (weak coupling)
    df_weak = df[df["tau"].apply(lambda t: t.imag > 3)]
    print(f"\nWeak coupling (Im(tau) > 3): {len(df_weak)} vacua")

## 6. Multiple runs with explicit IDs

The same sampling method can be run multiple times with different `run_id` labels.

In [ ]:
# Run alpha
z0a, tau0a, fl0a = sampler.initial_guesses_ISD(N=30, Nmax=10, mode="ISD+")
ma, ta, fa = solve_from_guesses(model_A, z0a, tau0a, fl0a)
with db.vacua_writer(model=model_A, method="ISD+", run_id="run_alpha") as w:
    w.append_batch(ma, ta, fa)
print(f"Run alpha: {w.count} vacua")

# Run beta (different random seed gives different starting points)
z0b, tau0b, fl0b = sampler.initial_guesses_ISD(N=30, Nmax=10, mode="ISD+")
mb, tb, fb = solve_from_guesses(model_A, z0b, tau0b, fl0b)
with db.vacua_writer(model=model_A, method="ISD+", run_id="run_beta") as w:
    w.append_batch(mb, tb, fb)
print(f"Run beta:  {w.count} vacua")

# Query by run_id
df_alpha = db.load_vacua(run_id="run_alpha")
df_beta = db.load_vacua(run_id="run_beta")
print(f"\nLoaded: alpha={len(df_alpha)}, beta={len(df_beta)}")

## 7. Designating vacua

Designation promotes session vacua (Tier 1) to permanent, validated records (Tier 2) with full provenance metadata. The `designate_vacua` method:

1. **Validates** each solution by checking $|D_I W| < $ `F_term_tol`
2. **Deduplicates** against existing designated solutions
3. **Records** label, author, timestamp, and optional notes
4. **Assigns** unique `designated_id` integers

Designated vacua survive `clear_cache()` unless explicitly included.

In [ ]:
# Load the ISD+ vacua from the first run
runs = db.query_vacua(method="ISD+")
if len(runs) > 0:
    first_run = runs.iloc[0]["run_id"]
    vacua_to_designate = db.load_vacua(run_id=first_run)
    
    if len(vacua_to_designate) > 0:
        # Designate with validation
        designated_ids = db.designate_vacua(
            vacua_to_designate.head(5),  # designate first 5
            label="NB25 tutorial demo",
            committed_by="tutorial",
            model=model_A,
            notes="Demonstration of vacua designation",
        )
        print(f"Designated {len(designated_ids)} vacua: IDs = {designated_ids}")
        
        # Query designated
        designated = db.query_designated(h12=2)
        print(f"\nDesignated catalog ({len(designated)} entries):")
        print(designated[["designated_id", "label", "committed_by"]].to_string())

In [ ]:
# Load full designated solution data
designated_data = db.load_designated(h12=2)
if len(designated_data) > 0:
    print(f"Loaded {len(designated_data)} designated solutions")
    print(f"Columns: {designated_data.columns.tolist()}")

## 8. Models from the HuggingFace database

We load two models from the TDF database and store vacua for each.

In [ ]:
from jaxvacua.database import TDFDatabase

try:
    db_tdf = TDFDatabase()
    
    # Query for two h11=2 models with GV data
    candidates = db_tdf.query(h11=2, has_gv=True)
    if len(candidates) >= 2:
        for idx in range(2):
            row = candidates.iloc[idx]
            ks_id = int(row["ks_id"])
            triang_id = int(row["triang_id"])
            
            model_tdf = db_tdf.load_model(
                ks_id=ks_id, triang_id=triang_id,
                include_gv=True, maximum_degree=2
            )
            sampler_tdf = jvc.data_sampler(model_tdf)
            
            z0, tau0, fl0 = sampler_tdf.initial_guesses_ISD(N=30, Nmax=10, mode="ISD+", moduli_sampling_mode="cone", ISD_oversample_factor=2, filter_moduli=True)
            m, t, f = solve_from_guesses(model_tdf, z0, tau0, fl0)
            
            if len(m) > 0:
                with db.vacua_writer(model=model_tdf, method="ISD+") as w:
                    w.append_batch(m, t, f)
                print(f"ks_id={ks_id}, triang_id={triang_id}: Stored {w.count} vacua")
            else:
                print(f"ks_id={ks_id}, triang_id={triang_id}: No converged vacua")
    else:
        print("Not enough h11=2 models with GV data in TDF database")
except Exception as e:
    print(f"TDF database not available: {e}")
    print("(This section requires huggingface-hub, pandas, pyarrow)")

## 9. Two local models (model_ID=1 vs model_ID=2)

Local models loaded via `model_ID` are automatically identified in the storage by `(h12, model_ID)`. This allows storing and querying vacua for different geometries within the same database.

- **Model A**: CP$[1,1,1,6,9]$ (`model_ID=1`) — the degree-18 hypersurface
- **Model B**: a different $h^{2,1}=2$ geometry (`model_ID=2`)

In [ ]:
# Model B: different geometry, same h12
model_B = jvc.FluxVacuaFinder(h12=2, model_ID=2, model_type="KS", maximum_degree=2)
sampler_B = jvc.data_sampler(model_B)

print(f"Model A: h12={model_A.h12}, model_ID={model_A.lcs_tree.model_ID}")
print(f"Model B: h12={model_B.h12}, model_ID={model_B.lcs_tree.model_ID}")

# Generate vacua for Model B
z0b, tau0b, fl0b = sampler_B.initial_guesses_ISD(N=50, Nmax=10, mode="ISD+")
mb, tb, fb = solve_from_guesses(model_B, z0b, tau0b, fl0b)
print(f"\nModel B: {len(mb)}/{len(z0b)} converged")

if len(mb) > 0:
    with db.vacua_writer(model=model_B, method="ISD+") as w:
        w.append_batch(mb, tb, fb)
    print(f"Stored {w.count} vacua for Model B")

In [ ]:
# Query across both models
all_h12_2 = db.query_vacua(h12=2)
print("All h12=2 runs:")
if "model_name" in all_h12_2.columns:
    print(all_h12_2[["model_name", "method", "count"]].to_string())
else:
    print(all_h12_2[["method", "count"]].to_string())

## 10. Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"Cleaned up {tmpdir}")